# RSNA Knee Abnormality Detection with MedDINOv3

This notebook demonstrates inference using the new model with MedDINOv3 backbone.
MedDINOv3 weights are frozen and used as a feature extractor.

## Setup and Imports

In [ ]:
# !pip install torch torchvision timm pydicom albumentations

In [ ]:
import os, cv2, math, random, warnings\nimport numpy as np\nimport pandas as pd\nimport pydicom\nfrom pathlib import Path\nfrom typing import List, Dict, Tuple, Optional\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.cuda.amp import autocast\nimport timm\nfrom tqdm import tqdm\n\n# Import the new model\nfrom model_with_meddinov3 import KneeAbnormalityModelWithMedDINOv3\n\nwarnings.filterwarnings('ignore')\nprint('✓ All imports successful')

## Configuration

In [ ]:
class MedDINOv3InfConfig:\n    '''Configuration for MedDINOv3-based inference'''\n    KAGGLE_DIR      = Path('/path/to/kaggle/data')\n    PYTHON_DIR      = Path('/path/to/project')\n    TEST_SERIES_CSV = KAGGLE_DIR / 'test_series.csv'\n    TEST_IMG_DIR    = KAGGLE_DIR / 'test_series'\n    CHECKPOINT      = str(PYTHON_DIR / 'best_model_meddinov3.pth')\n    SUBMISSION_PATH = str(KAGGLE_DIR / 'submission.csv')\n\n    # Image processing\n    PATCH_SIZE   = 224\n    PATCH_STRIDE = 112\n    MAX_SLICES   = 12\n\n    # Model labels and parameters\n    LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',\n              'Medial OA','Lateral OA','PF OA','Effusion',\n              'Synovitis',\"Baker's\",'Contusion','Fracture']\n    NUM_LABELS   = 12\n    PLANE_TO_IDX = {'Sagittal': 0, 'Axial': 1, 'Coronal': 2}\n    NUM_PLANES   = 3\n\n    # MedDINOv3 uses 768-d features (ViT-B)\n    SLICE_DIM    = 512\n    META_DIM     = 256\n    PATCH_ENC_HEADS  = 8\n    PATCH_ENC_LAYERS = 2\n    SLICE_ENC_HEADS  = 8\n    SLICE_ENC_LAYERS = 2\n    DROPOUT      = 0.3\n\n    # Inference settings\n    BATCH_SIZE   = 4\n    NUM_WORKERS  = 4\n    USE_TTA      = True\n    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n\nicfg = MedDINOv3InfConfig()\nprint(f'Device: {icfg.DEVICE}')

## Load Model

In [ ]:
print('Loading model with MedDINOv3 backbone...')\nmodel = KneeAbnormalityModelWithMedDINOv3(icfg).to(icfg.DEVICE)\n\n# Load checkpoint if available\nif os.path.exists(icfg.CHECKPOINT):\n    ckpt = torch.load(icfg.CHECKPOINT, map_location='cpu')\n    model.load_state_dict(ckpt['model_state_dict'], strict=True)\n    model.eval()\n    print(f'✓ Checkpoint loaded: epoch={ckpt.get(\"epoch\",\"?\")} val_auc={ckpt.get(\"val_auc\", float(\"nan\")):.4f}')\nelse:\n    print(f'⚠ Checkpoint not found at {icfg.CHECKPOINT}')\n    model.eval()\n\n# Show model info\ntotal_params = sum(p.numel() for p in model.parameters())\nfrozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)\nprint(f'✓ Model loaded successfully')\nprint(f'  Total parameters: {total_params:,}')\nprint(f'  Frozen (MedDINOv3): {frozen_params:,}')\nprint(f'  Trainable: {total_params - frozen_params:,}')

## Run Inference\n\nUse your existing dataset loader code here.

In [ ]:
@torch.no_grad()\ndef run_inference(model, loader, device):\n    '''Run inference on test data'''\n    model.eval()\n    study_ids, probs = [], []\n    \n    for batch in tqdm(loader, desc='Inference'):\n        patches = batch['patches'].to(device)\n        slice_mask = batch['slice_mask'].to(device)\n        patch_valid = batch['patch_valid'].to(device)\n        plane_id = batch['plane_id'].to(device)\n        fluid_sensitive = batch['fluid_sensitive'].to(device)\n        fat_suppression = batch['fat_suppression'].to(device)\n        \n        logits = model(patches, slice_mask, patch_valid, plane_id,\n                      fluid_sensitive, fat_suppression)\n        \n        probs.append(torch.sigmoid(logits).cpu().numpy())\n        study_ids.extend(batch['study_id'])\n    \n    return study_ids, np.vstack(probs)\n\n# Example usage (uncomment when data is available):\n# study_ids, probs = run_inference(model, test_loader, icfg.DEVICE)\n# print(f'\\nPredictions generated for {len(study_ids)} studies')

## Key Differences from ResNet50\n\n- **Backbone**: ViT-B (Transformer) instead of ResNet50 (CNN)\n- **Feature Dimension**: 768-d instead of 2048-d\n- **Frozen Weights**: MedDINOv3 backbone is completely frozen during training\n- **Training Focus**: Only downstream layers (projections, encoders, classifier) are trained\n- **Medical Pretraining**: Benefits from 3.8M CT images in MedDINOv3 pretraining